Generate a simple GENAi app using langchain and OpenAI

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [4]:
## Data Ingestion - From the website we want to scrape the data

from langchain_community.document_loaders import WebBaseLoader

In [5]:
url = "https://docs.langchain.com/langsmith/trace-with-langchain"
loader=WebBaseLoader(url)
loader

In [6]:
docs = loader.load()
docs[0].page_content[:500]  # Print the first 500 characters of the page content

"Trace LangChain applications (Python and JS/TS) - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAgent frameworksTrace LangChai"

In [7]:
## Load Data --> Docs --> Divide our text into chunks --> Create embeddings or vectors for each chunk --> Store the embeddings in a vector database
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents_split = recursive_splitter.split_documents(docs)
documents_split[0].page_content[:500]  # Print the first 500 characters of the first chunk





"Trace LangChain applications (Python and JS/TS) - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAgent frameworksTrace LangChai"

In [8]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

FAISS is created by Facebook and is a library for efficient similarity search and clustering of dense vectors. It is widely used in machine learning and information retrieval applications to perform tasks such as nearest neighbor search, clustering, and dimensionality reduction on large datasets.

In [10]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents_split, embeddings)
vectorstore

In [11]:
query="How to trace with LangChain?"
result=vectorstore.similarity_search(query, k=2)  # Retrieve the top 2 most similar documents to the query
result[0].page_content[:500]  # Print the first 500 characters of the most similar document

'LangChain objects inside traceable (JS only)Tracing LangChain child runs via traceable / RunTree API (JS only)Tracing setupIntegrationsAgent frameworksTrace LangChain applications (Python and JS/TS)Copy pageCopy pageCopy pageCopy pageLangSmith integrates seamlessly with LangChain (Python and JavaScript), the popular open-source framework for building LLM applications.'

In [ ]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model="gpt-4o")
llm
# llm = ChatOpenAI(model="gpt-4.1-mini") 
# use this cheaper

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000026E1F2C2600>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000026E1F2B5C10>, root_client=<openai.OpenAI object at 0x

Retrieval chain - We can use the vectorstore to retrieve relevant documents based on a query and then use those documents to generate a response using a language model.

Document chaining - We can use the retrieved documents to answer the query. We can use a language model to generate a response based on the retrieved documents.

document_chain

is waiting for:

Context (documents)
Question

Only then does it produce an answer.

In [14]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
Answer the following question based only on the provided context.

<context>
{context}
</context>

Question:
{input}
"""
)
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context.\n\n<context>\n{context}\n</context>\n\nQuestion:\n{input}\n'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_

In [15]:
from langchain_core.documents import Document
document_chain.invoke({
    "input": query,
    "context":[Document(page_content="LangChain objects inside traceable (JS only)Tracing LangChain child runs via traceable")]

})

"To trace with LangChain, you need to utilize the functionality for tracing child runs within the LangChain library, specifically for JavaScript (JS) as mentioned in the provided context. This typically involves setting up tracing to monitor and log the execution of various operations and components within LangChain. However, the context provided only gives a brief mention of 'traceable' in relation to JavaScript and does not provide specific steps or methods for setting up or executing tracing. For comprehensive instructions, you would generally refer to the LangChain documentation or relevant guides that provide detailed procedures for implementing tracing in your specific use case."

- ❓ Input = User's question
- 🔍 Retriever = Finds relevant documents
- 🗄️ Vector Store DB = Stores document embeddings
- 📄 Output of Retriever = Most relevant documents
- 🤖 LLM = Generates the final answer using those documents

What does the Retriever do?

When you call:

```python
retriever = vectorstore.as_retriever()
docs = retriever.invoke("What is AI?")
```

Internally it does:

- 🔄 Convert the question into an embedding.
- 🔎 Perform similarity search in the vector database.
- 📄 Return the top matching documents.

So you don't need to call:

vectorstore.similarity_search(query)

In [17]:
retriever=vectorstore.as_retriever()
from langchain_classic.chains import create_retrieval_chain

retrieval_chain=create_retrieval_chain(retriever,document_chain)
retrieval_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026E1F21D370>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context.\n\n<context>\n{context}\n</context>\n\nQuestion:\n{input}\n'), 

In [18]:
# Get the response from the LLM

response = retrieval_chain.invoke(
    {"input":query}
)
response['answer']

'To trace with LangChain, you should follow these general steps:\n\n1. **Configure Your Environment**: Set up the necessary environment configuration. Optionally, specify a custom project by setting the `LANGSMITH_PROJECT` environment variable if you don\'t want to use the default project. This can be set with the command:\n   ```bash\n   export LANGSMITH_PROJECT=my-project\n   ```\n\n2. **Set Up Tracing**: Utilize the LangChain tracing tools such as `LangChainTracer()` in your application. For example, in JavaScript:\n   ```javascript\n   const tracer = new LangChainTracer();\n   await chain.invoke(\n     {\n       question: "Am I using a callback?",\n       context: "I\'m using a callback"\n     },\n     { callbacks: [tracer] }\n   );\n   ```\n\n3. **Log the Trace**: Ensure your application is properly logging traces, either statically by configuring the environment ahead of execution, or dynamically by passing tracer objects like shown above.\n\n4. **View and Analyze Traces**: After

In [19]:
response

{'input': 'How to trace with LangChain?',
 'context': [Document(id='17e69d7e-fb92-42b4-88fd-1f6e061b6693', metadata={'source': 'https://docs.langchain.com/langsmith/trace-with-langchain', 'title': 'Trace LangChain applications (Python and JS/TS) - Docs by LangChain', 'language': 'en'}, page_content='LangChain objects inside traceable (JS only)Tracing LangChain child runs via traceable / RunTree API (JS only)Tracing setupIntegrationsAgent frameworksTrace LangChain applications (Python and JS/TS)Copy pageCopy pageCopy pageCopy pageLangSmith integrates seamlessly with LangChain (Python and JavaScript), the popular open-source framework for building LLM applications.'),
  Document(id='12ba2f79-aff2-4ac9-bf60-a6be6e871532', metadata={'source': 'https://docs.langchain.com/langsmith/trace-with-langchain', 'title': 'Trace LangChain applications (Python and JS/TS) - Docs by LangChain', 'language': 'en'}, page_content='const tracer = new LangChainTracer();\nawait chain.invoke(\n  {\n    question

In [20]:
response['context']

[Document(id='17e69d7e-fb92-42b4-88fd-1f6e061b6693', metadata={'source': 'https://docs.langchain.com/langsmith/trace-with-langchain', 'title': 'Trace LangChain applications (Python and JS/TS) - Docs by LangChain', 'language': 'en'}, page_content='LangChain objects inside traceable (JS only)Tracing LangChain child runs via traceable / RunTree API (JS only)Tracing setupIntegrationsAgent frameworksTrace LangChain applications (Python and JS/TS)Copy pageCopy pageCopy pageCopy pageLangSmith integrates seamlessly with LangChain (Python and JavaScript), the popular open-source framework for building LLM applications.'),
 Document(id='12ba2f79-aff2-4ac9-bf60-a6be6e871532', metadata={'source': 'https://docs.langchain.com/langsmith/trace-with-langchain', 'title': 'Trace LangChain applications (Python and JS/TS) - Docs by LangChain', 'language': 'en'}, page_content='const tracer = new LangChainTracer();\nawait chain.invoke(\n  {\n    question: "Am I using a callback?",\n    context: "I\'m using a